# Simple RAG Chatbot — 100% Open Source (Hugging Face)

This notebook builds a Retrieval-Augmented Generation (RAG) chatbot using only
**open source models from Hugging Face** — no OpenAI, no paid APIs.

| Component | Model | Type |
|---|---|---|
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` | Open source, runs on CPU |
| Vector store | `FAISS` (Facebook AI Similarity Search) | Open source library |
| LLM (generator) | `Qwen/Qwen2.5-1.5B-Instruct` | Open source, non-gated, small enough for CPU/free-tier GPU |

Each step is broken into its own code block, in order:

1. Install dependencies
2. Load & chunk the document(s)
3. Convert chunks to embeddings
4. Build a vector index (FAISS) for retrieval
5. Load the open-source LLM
6. Define the retrieval function
7. Define the RAG prompt + generation function
8. Chat loop — ask questions!

> 💡 You can swap `Qwen/Qwen2.5-1.5B-Instruct` for any other open Hugging Face model
> (e.g. `microsoft/Phi-3-mini-4k-instruct`, `HuggingFaceH4/zephyr-7b-beta`,
> `mistralai/Mistral-7B-Instruct-v0.3` if you have more GPU memory).

## Step 1 — Install dependencies
All libraries below are open source (Hugging Face `transformers`, `sentence-transformers`, `faiss-cpu`).

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate torch langchain-text-splitters pypdf


## Step 2 — Load and chunk the document(s)

RAG works best on small, semantically coherent chunks rather than whole documents.
Here we:
- Load one or more documents (`.txt` or `.pdf`) from a `docs/` folder
- Split each document into overlapping chunks using a simple recursive character splitter

**Where does `docs/` need to be?** That depends on where this notebook is running:

| Environment | What to do |
|---|---|
| **Google Colab** (working dir shows as `/content`) | Run the cell below — it detects Colab automatically and pops up a file-upload dialog. Uploaded files land in `docs/` on the Colab VM. |
| **Local Jupyter** (on your own machine) | Just create a `docs` folder next to this `.ipynb` file and drop your `.txt`/`.pdf` files in it — no upload step needed. |

If no files are found either way, three short sample documents (on RAG, IoT, and vector DBs) are used automatically so the rest of the notebook still runs end-to-end.

In [ ]:
import os

DOCS_FOLDER = "docs"
os.makedirs(DOCS_FOLDER, exist_ok=True)


**Using Google Drive instead of re-uploading every session?** Uncomment and run the
cell below *before* the upload cell — it mounts your Drive and points `DOCS_FOLDER`
at a folder there, so files persist across Colab sessions. Skip it entirely if you're
fine re-uploading each time.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# DOCS_FOLDER = "/content/drive/MyDrive/docs"  # upload your docs there via the Drive UI
# os.makedirs(DOCS_FOLDER, exist_ok=True)


In [ ]:
# --- Auto-detect Google Colab and offer a file upload into DOCS_FOLDER ---
# (Skipped automatically if DOCS_FOLDER already has files — e.g. if you mounted Drive above.)
try:
    import google.colab  # only importable inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.listdir(DOCS_FOLDER):
    from google.colab import files
    print("Running in Colab — pick one or more .txt / .pdf files to upload:")
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, os.path.join(DOCS_FOLDER, fname))

print(f"IN_COLAB={IN_COLAB} | Files in '{DOCS_FOLDER}':", os.listdir(DOCS_FOLDER))


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Load .txt and .pdf files from DOCS_FOLDER ---
documents = []

if os.path.isdir(DOCS_FOLDER):
    for fname in sorted(os.listdir(DOCS_FOLDER)):
        fpath = os.path.join(DOCS_FOLDER, fname)
        if fname.lower().endswith(".txt"):
            with open(fpath, "r", encoding="utf-8") as f:
                documents.append(f.read())
        elif fname.lower().endswith(".pdf"):
            from pypdf import PdfReader
            reader = PdfReader(fpath)
            text = "\n".join(page.extract_text() or "" for page in reader.pages)
            documents.append(text)

# --- Fallback sample documents (used if docs/ is empty) ---
if not documents:
    documents = [
        """
        Retrieval-Augmented Generation (RAG) is a technique that combines information
        retrieval with text generation. Instead of relying only on what a language model
        learned during training, RAG retrieves relevant passages from an external knowledge
        base at query time and feeds them to the model as context. This reduces
        hallucination and lets the model answer questions about documents it has never
        seen during training.
        """,
        """
        The Internet of Things (IoT) refers to the network of physical devices, vehicles,
        appliances, and other objects embedded with sensors, software, and connectivity
        that allows them to collect and exchange data. Common IoT applications include
        smart homes, industrial monitoring, wearable health devices, and smart city
        infrastructure such as traffic sensors and connected utility meters.
        """,
        """
        Vector databases store data as high-dimensional numerical embeddings and allow
        fast similarity search. Popular open source options include FAISS, Chroma, and
        Qdrant. In a RAG pipeline, a vector database stores the embeddings of document
        chunks so the system can quickly retrieve the chunks most relevant to a user's
        question.
        """,
    ]

print(f"Loaded {len(documents)} document(s).")

# --- Chunk each document ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,      # characters per chunk
    chunk_overlap=50,    # overlap between chunks to preserve context
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in documents:
    doc_chunks = splitter.split_text(doc.strip())
    chunks.extend(doc_chunks)

print(f"Split into {len(chunks)} chunks.")
print("\nExample chunk:\n", chunks[0])


## Step 3 — Convert chunks to embeddings

We use `sentence-transformers/all-MiniLM-L6-v2` — a small, fast, fully open-source
embedding model (384-dimensional vectors) that runs comfortably on CPU.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True,  # normalize so we can use cosine similarity via inner product
)

print("Embedding matrix shape:", chunk_embeddings.shape)


## Step 4 — Build a vector index with FAISS

FAISS (open source, by Meta AI) lets us do fast nearest-neighbor search over the
chunk embeddings. Since embeddings are normalized, we use an inner-product index,
which is equivalent to cosine similarity search.

In [ ]:
import faiss

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # inner product = cosine similarity (vectors are normalized)
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}.")


## Step 5 — Load the open-source LLM

`Qwen/Qwen2.5-1.5B-Instruct` is a small, fully open-source, non-gated instruction-tuned
model from Alibaba's Qwen team — a good balance of quality and speed for a CPU/small-GPU demo.

Swap the `LLM_MODEL_NAME` below for any other open Hugging Face model if you have more
compute available (e.g. `microsoft/Phi-3-mini-4k-instruct`, `HuggingFaceH4/zephyr-7b-beta`).

In [ ]:
import torch
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, logging as hf_logging

# Quiet down noisy-but-harmless info/warning logs from transformers
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    clean_up_tokenization_spaces=False,  # correct default for this BPE tokenizer; silences the warning
)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

# The model's default generation_config sets max_length=20, which conflicts with
# max_new_tokens below. Clearing it lets max_new_tokens take precedence silently.
model.generation_config.max_length = None

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,   # greedy decoding — more literal/grounded, less likely to wander into outside knowledge
)

print(f"Loaded LLM: {LLM_MODEL_NAME} on {'GPU' if torch.cuda.is_available() else 'CPU'}")


## Step 6 — Define the retrieval function

Given a user question, embed it with the same embedding model, then search the FAISS
index for the top-k most similar chunks.

In [ ]:
def retrieve(query: str, top_k: int = 3):
    """Return the top_k most relevant chunks for a query."""
    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )
    scores, indices = index.search(query_embedding, top_k)
    results = [(chunks[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]
    return results

# quick test
for chunk, score in retrieve("What is IoT?", top_k=2):
    print(f"[score={score:.3f}] {chunk[:120]}...\n")


## Step 7 — Build the RAG prompt and generate an answer

We stuff the retrieved chunks into the prompt as context, then ask the LLM to answer
using only that context (to reduce hallucination).

**Note on faithfulness:** smaller models (like the 1.5B one here) sometimes blend in
their own pretrained knowledge alongside the context, especially for well-known topics.
The prompt below uses a `system` instruction, repeats the constraint after the context
(models weight text closer to the question more heavily), and generation uses greedy
decoding (Step 5) to stay as literal as possible. If you still see leakage, swapping in
a larger open model (e.g. `microsoft/Phi-3-mini-4k-instruct`) will follow this kind of
instruction more reliably.

In [ ]:
def build_prompt(query: str, retrieved_chunks: list):
    context = "\n\n".join([f"- {c}" for c, _ in retrieved_chunks])

    system_msg = (
        "You are a strict retrieval-based assistant. You must answer ONLY using the "
        "context the user provides in their message. Do not use any outside knowledge, "
        "even if you already know the answer. If the context does not contain the answer, "
        "respond with exactly: \"I don't have enough information to answer that.\" "
        "Do not add facts, names, or details that are not explicitly present in the context."
    )

    user_msg = f"""Context:
{context}

Question: {query}

Reminder: answer using ONLY the context above. Do not use any prior/outside knowledge."""

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]


def rag_answer(query: str, top_k: int = 3) -> str:
    retrieved = retrieve(query, top_k=top_k)
    messages = build_prompt(query, retrieved)

    chat_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = generator(chat_prompt, return_full_text=False)[0]["generated_text"]
    return output.strip()


# quick test
answer = rag_answer("What is Retrieval-Augmented Generation?")
print(answer)


## Step 8 — Chat with your documents

Run the cell below and type questions about your document(s). Type `exit` or `quit` to stop.

In [ ]:
while True:
    user_query = input("You: ")
    if user_query.strip().lower() in {"exit", "quit"}:
        print("Goodbye!")
        break
    response = rag_answer(user_query)
    print(f"Bot: {response}\n")


## Notes & next steps

- **All models are open source:** embeddings (`all-MiniLM-L6-v2`), LLM (`Qwen2.5-1.5B-Instruct`), and vector search (FAISS) — no paid API keys required.
- **Scaling up:** for higher-quality answers, swap in a bigger open model (`Phi-3-mini`, `Mistral-7B-Instruct`, `Zephyr-7B`) — just change `LLM_MODEL_NAME` in Step 5 (more GPU memory needed).
- **More documents:** drop `.txt` files into a `docs/` folder next to this notebook — Step 2 will pick them up automatically. PDFs can be added with `pypdf` + a small loader function.
- **Persisting the index:** call `faiss.write_index(index, "index.faiss")` to save it and `faiss.read_index(...)` to reload later instead of re-embedding every run.
- **Better chunking:** tune `chunk_size` / `chunk_overlap` in Step 2 based on your documents' structure.